In [2]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tabpfn import TabPFNClassifier
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, confusion_matrix
import argparse
import os
import time
import json

from utils.helper import preprocess_mimic_data_advanced, preprocess_eicu_data_advanced, get_class_weights, quantify_dataset_imbalance, convert_to_serializable
from data.dataset import TabularDataset
from config import OUTPUT_PATH, VAL_SIZE, TEST_SIZE, TARGET_COL, RESULT_PATH, CUTOFF, EICU_TARGETS, EICU_FILE

# Start timing
overall_start_time = time.time()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
dataset_flag = "mimic"  # or "eicu"
experiment_id = "1"  # Unique identifier for the experiment
dataset_flag =  "mimic"  # or "eicu"
target_col =    TARGET_COL
weighting_strategy =    "effective"  # or "none"



In [4]:
RANDOM_SEED = 42

# ---------------------
# Filter sizes
# ---------------------
FILTER_SIZE = [300, 1000, 1500, 3500, 4500, 6000, 8000, 12000] if dataset_flag == 'eicu' else \
              [500,750,1000,1250,1500,1750,2000,2250,2500,2750,3000,3250,3500,3750,4000]
FILTER_SIZE.reverse()

print(f"Running experiments with filter sizes: {FILTER_SIZE}")

all_results = {}

for filter_size in FILTER_SIZE:
    print(f"\n===== Running experiment for FILTER_SIZE = {filter_size} =====")

    # ---------------------
    # Dataset processing
    # ---------------------
    if dataset_flag == 'mimic':
        print("Processing MIMIC dataset...")
        processed_data = preprocess_mimic_data_advanced(
            output_path=OUTPUT_PATH,
            filename=CUTOFF if CUTOFF else 'mimic_multimodal_image_centric_streamlined_found.csv',
            filter_size=filter_size,
            target_col=target_col,
            impute_missing=True,
            staging=False,
            test_size=TEST_SIZE,
            val_size=VAL_SIZE,
            random_state=RANDOM_SEED
        )
        X_train = processed_data['X_train']
        X_val = processed_data['X_val']
        X_test = processed_data['X_test']
        y_train = processed_data['y_train']
        y_val = processed_data['y_val']
        y_test = processed_data['y_test']

        # Drop irrelevant columns
        cols_to_drop = {
            'diagnosis': ['icd_code_broad', 'disposition_grouped'],
            'disposition_grouped': ['icd_code_broad', 'diagnosis'],
            'icd_code_broad': ['diagnosis', 'disposition_grouped']
        }.get(target_col, [])
        for df in [X_train, X_val, X_test]:
            df.drop(columns=cols_to_drop + ['path'], inplace=True, errors='ignore')
    else:
        print("Processing eICU dataset...")
        processed_data = preprocess_eicu_data_advanced(
            output_path=OUTPUT_PATH,
            filename=EICU_FILE,
            target_col=target_col,
            impute_missing=True,
            filter_size=filter_size,
            test_size=TEST_SIZE,
            val_size=VAL_SIZE,
            random_state=RANDOM_SEED
        )
        X_train = processed_data['X_train'].apply(pd.to_numeric, errors="coerce").fillna(0).astype(np.float32)
        X_val = processed_data['X_val'].apply(pd.to_numeric, errors="coerce").fillna(0).astype(np.float32)
        X_test = processed_data['X_test'].apply(pd.to_numeric, errors="coerce").fillna(0).astype(np.float32)
        y_train = processed_data['y_train']
        y_val = processed_data['y_val']
        y_test = processed_data['y_test']

    # ---------------------
    # Class weights & imbalance
    # ---------------------
    class_weights, unique_classes = get_class_weights(y_train, strategy=weighting_strategy, beta=0.9999)
    class_counts = [np.sum(y_train == cls) for cls in unique_classes]

    print("Class weights:", class_weights)
    imbalance_metrics = quantify_dataset_imbalance(class_counts=class_counts, class_weights=class_weights)
    print(f"Imbalance metrics ({weighting_strategy}): {imbalance_metrics}")

Running experiments with filter sizes: [4000, 3750, 3500, 3250, 3000, 2750, 2500, 2250, 2000, 1750, 1500, 1250, 1000, 750, 500]

===== Running experiment for FILTER_SIZE = 4000 =====
Processing MIMIC dataset...
STARTING MIMIC DATA PREPROCESSING PIPELINE
Step 1: Loading data...
Loaded data shape: (30306, 313)

Step 2: Dropping unnecessary columns...
Columns not found (skipped): ['dicom_id', 'subject_id', 'study_id', 'stay_id', 'hadm_id', 'icd_code', 'icd_title', 'icd_version', 'disposition']
Data shape after dropping columns: (30306, 313)
Note: 'path' column retained for output data frames

Step 3: Handling missing values...
Imputation strategy: Impute

Step 4: Analyzing target variable distribution...
Class distribution for 'icd_code_broad':
  786: 11387 (37.6%)
  780: 9340 (30.8%)
  486: 4932 (16.3%)
  789: 4647 (15.3%)

Step 5: Feature preprocessing...
Encoding columns not found (skipped): ['gender', 'race', 'arrival_transport', 'pain']
Class count before subsampling
 icd_code_broad


In [ ]:
# !pip install "tabpfn-extensions[many_class]"

In [5]:

# !pip install "tabpfn-extensions[all] @ git+https://github.com/PriorLabs/tabpfn-extensions.git"

In [6]:
from tabpfn_extensions.manyclass_classifier import ManyClassClassifier

# ---------------------
# TabPFN training & evaluation
# ---------------------
print(f"\nTraining TabPFNClassifier for FILTER_SIZE={filter_size} ...")
start_time = time.time()
# src: https://github.com/PriorLabs/tabpfn-extensions/blob/main/examples/many_class/many_class_classifier_example.py
# clf = TabPFNClassifier(balance_probabilities=True,ignore_pretraining_limits=True,n_preprocessing_jobs=4)  # defaults to v2.5
# clf.to(device)
estimator = TabPFNClassifier(device="cuda")
clf = ManyClassClassifier(estimator=estimator)

clf.fit(X_train.to_numpy(), y_train.to_numpy())
# preds = clf.predict(X_train.to_numpy())
training_time = time.time() - start_time

ModuleNotFoundError: No module named 'pkg_resources'